# 📒 Notebook 4 — Feature Engineering
> **Peran:** Senior Data Scientist | **Proyek:** SPK Pantau Pasar Sumbawa

---

## 🧩 1. Ikhtisar Teknik Feature Engineering

Dataset harga bahan pokok memerlukan representasi temporal yang kaya agar Random Forest Regressor dapat belajar pola musiman, tren, dan volatilitas harga. Kami menerapkan empat kelompok fitur:

| Kelompok | Fitur | Deskripsi |
|----------|-------|-----------|
| **Imputasi** | `Harga` (ffill) | Isi hari libur/akhir pekan dengan harga terakhir per komoditas. Nilai awal sebelum observasi dibiarkan NaN untuk menghindari target leakage, lalu dibersihkan di akhir. |
| **Kalender** | `Tahun`, `Bulan`, `Hari`, `DayOfWeek`, `Quarter`, `WeekOfYear` | Konteks waktu absolut dan siklus kalender |
| **Lag** | `Harga_Kemarin`, `Harga_Minggu_Lalu` | Harga historis pendek per komoditas |
| **Rolling** | `Rolling_Mean_7`, `Rolling_Mean_14`, `Rolling_Std_7`, `Rolling_Max_7`, `Rolling_Min_7` | Statistik jendela geser per komoditas |

---

## 📋 2. Kenapa Forward Fill?

Pasar tidak beroperasi setiap hari. Pada hari libur/akhir pekan, harga tidak berubah — pedagang masih menjual di harga hari terakhir. Strategi **forward fill per komoditas** merefleksikan realitas ini:
```
Hari 1 (Senin)  : Harga = 14.000 ← data asli
Hari 2 (Selasa) : Harga = NaN    ← forward fill → 14.000
Hari 3 (Rabu)   : Harga = NaN    ← forward fill → 14.000
Hari 4 (Kamis)  : Harga = 14.500 ← data asli
```


In [1]:
import logging
from pathlib import Path
import pandas as pd
import numpy as np

# ── Logging ───────────────────────────────────────────────────────────────────
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S",
)
logger = logging.getLogger("FeatureEngineering")

# ── Paths ─────────────────────────────────────────────────────────────────────
PROCESSED_DIR: Path = Path("../processed_data/processed")
FEATURES_DIR:  Path = Path("../processed_data/features")
FEATURES_DIR.mkdir(parents=True, exist_ok=True)

# ── Load Preprocessed Dataset ─────────────────────────────────────────────────
logger.info("Memuat preprocessed dataset …")
df: pd.DataFrame = pd.read_csv(
    PROCESSED_DIR / "preprocessed_dataset.csv", parse_dates=["Tanggal"]
)
logger.info("Dataset dimuat: %s baris, %s kolom", *df.shape)


10:44:32 [INFO] Memuat preprocessed dataset …
10:44:32 [INFO] Dataset dimuat: 121980 baris, 4 kolom


## 🩹 3. Imputasi Missing Value


In [2]:
# ── Sort chronologically per commodity ───────────────────────────────────────
df.sort_values(["Komoditas", "Tanggal"], inplace=True)
df.reset_index(drop=True, inplace=True)

# ── Forward fill then backward fill per commodity ─────────────────────────────
logger.info("Imputasi harga (ffill) per komoditas …")
df["Harga"] = df.groupby("Komoditas")["Harga"].ffill()

nan_remaining: int = df["Harga"].isna().sum()
logger.info("NaN tersisa setelah imputasi: %d", nan_remaining)


10:44:32 [INFO] Imputasi harga (ffill) per komoditas …
10:44:32 [INFO] NaN tersisa setelah imputasi: 44819


## 📆 4. Ekstraksi Fitur Kalender


In [3]:
logger.info("Ekstraksi fitur kalender …")

df["Tahun"]     = df["Tanggal"].dt.year
df["Bulan"]     = df["Tanggal"].dt.month
df["Hari"]      = df["Tanggal"].dt.day
df["DayOfWeek"] = df["Tanggal"].dt.dayofweek          # 0 = Senin, 6 = Minggu
df["Quarter"]   = df["Tanggal"].dt.quarter
df["WeekOfYear"]= df["Tanggal"].dt.isocalendar().week.astype(int)

print("Fitur kalender berhasil ditambahkan:")
print(df[["Tanggal", "Tahun", "Bulan", "Hari", "DayOfWeek",
          "Quarter", "WeekOfYear"]].head())


10:44:32 [INFO] Ekstraksi fitur kalender …


Fitur kalender berhasil ditambahkan:
     Tanggal  Tahun  Bulan  Hari  DayOfWeek  Quarter  WeekOfYear
0 2021-01-01   2021      1     1          4        1          53
1 2021-01-02   2021      1     2          5        1          53
2 2021-01-03   2021      1     3          6        1          53
3 2021-01-04   2021      1     4          0        1           1
4 2021-01-05   2021      1     5          1        1           1


## ⏳ 5. Ekstraksi Lag Features

Lag features menangkap ketergantungan temporal jangka pendek. Dihitung **per komoditas** sehingga nilai lag tidak bercampur antar komoditas.


In [4]:
logger.info("Ekstraksi lag features …")

df["Harga_Kemarin"]    = df.groupby("Komoditas")["Harga"].shift(1)
df["Harga_Minggu_Lalu"]= df.groupby("Komoditas")["Harga"].shift(7)

print("Lag features (5 baris pertama beras_medium):")
mask_beras = df["Komoditas"] == "beras_medium"
print(df.loc[mask_beras, ["Tanggal", "Harga", "Harga_Kemarin", "Harga_Minggu_Lalu"]].head(10))


10:44:32 [INFO] Ekstraksi lag features …


Lag features (5 baris pertama beras_medium):
        Tanggal    Harga  Harga_Kemarin  Harga_Minggu_Lalu
8560 2021-01-01      NaN            NaN                NaN
8561 2021-01-02      NaN            NaN                NaN
8562 2021-01-03      NaN            NaN                NaN
8563 2021-01-04  10000.0            NaN                NaN
8564 2021-01-05  10000.0        10000.0                NaN
8565 2021-01-06  10000.0        10000.0                NaN
8566 2021-01-07  10000.0        10000.0                NaN
8567 2021-01-08  10000.0        10000.0                NaN
8568 2021-01-09  10000.0        10000.0                NaN
8569 2021-01-10  10000.0        10000.0                NaN


## 📈 6. Ekstraksi Rolling Features

Rolling window dihitung **per komoditas** dengan `groupby().transform()`.  
Window 7 = seminggu ke belakang. Window 14 = dua minggu ke belakang.


In [5]:
logger.info("Ekstraksi rolling features (window 7 & 14) …")

def rolling_stat(series: pd.Series, window: int, stat: str) -> pd.Series:
    """Compute a rolling statistic on a Series.

    Args:
        series: Input price series (per commodity group).
        window: Rolling window size in days.
        stat: One of 'mean', 'std', 'max', 'min'.

    Returns:
        Rolling statistic Series.
    """
    r = series.rolling(window=window)
    return getattr(r, stat)()


grouped = df.groupby("Komoditas")["Harga"]

df["Rolling_Mean_7"]  = grouped.transform(lambda x: rolling_stat(x.shift(1), 7,  "mean"))
df["Rolling_Mean_14"] = grouped.transform(lambda x: rolling_stat(x.shift(1), 14, "mean"))
df["Rolling_Std_7"]   = grouped.transform(lambda x: rolling_stat(x.shift(1), 7,  "std"))
df["Rolling_Max_7"]   = grouped.transform(lambda x: rolling_stat(x.shift(1), 7,  "max"))
df["Rolling_Min_7"]   = grouped.transform(lambda x: rolling_stat(x.shift(1), 7,  "min"))

logger.info("Rolling features berhasil ditambahkan.")
print("Kolom akhir dataset:")
print(df.columns.tolist())


10:44:32 [INFO] Ekstraksi rolling features (window 7 & 14) …
10:44:32 [INFO] Rolling features berhasil ditambahkan.


Kolom akhir dataset:
['Tanggal', 'Komoditas', 'Harga', 'Sumber', 'Tahun', 'Bulan', 'Hari', 'DayOfWeek', 'Quarter', 'WeekOfYear', 'Harga_Kemarin', 'Harga_Minggu_Lalu', 'Rolling_Mean_7', 'Rolling_Mean_14', 'Rolling_Std_7', 'Rolling_Max_7', 'Rolling_Min_7']


## ✅ 7. Validasi Kualitas Data


In [6]:
logger.info("Menjalankan validasi kualitas …")

# ── 1. Drop rows with NaN in target or features to prevent target leakage ─────
feature_cols = [
"Harga", "Harga_Kemarin", "Harga_Minggu_Lalu",
"Rolling_Mean_7", "Rolling_Mean_14", "Rolling_Std_7",
"Rolling_Max_7", "Rolling_Min_7"
]
logger.info("Menghapus baris dengan NaN pada target/fitur (akibat ffill-only & lag/rolling) …")
df.dropna(subset=feature_cols, inplace=True)
logger.info("Baris tersisa setelah pembersihan: %d", len(df))


# ── 2. Uniqueness check ───────────────────────────────────────────────────────
dup_count: int = df.duplicated(subset=["Tanggal", "Komoditas"]).sum()
assert dup_count == 0, f"Ditemukan {dup_count} duplikat pada ['Tanggal', 'Komoditas']!"
print(f"✅ Duplikat ['Tanggal','Komoditas']: {dup_count}  (OK)")

# ── 3. Missing values summary ─────────────────────────────────────────────────
null_summary = df.isnull().sum()
print("\nJumlah nilai NaN per kolom:")
print(null_summary[null_summary > 0].to_string() or "  Tidak ada NaN.")

# ── 4. Describe ───────────────────────────────────────────────────────────────
print("\nStatistik deskriptif fitur numerik:")
print(df.describe().round(2))

# ── 5. Info ───────────────────────────────────────────────────────────────────
print("\nInfo DataFrame:")
df.info()


10:44:32 [INFO] Menjalankan validasi kualitas …
10:44:32 [INFO] Menghapus baris dengan NaN pada target/fitur (akibat ffill-only & lag/rolling) …
10:44:32 [INFO] Baris tersisa setelah pembersihan: 76363


✅ Duplikat ['Tanggal','Komoditas']: 0  (OK)

Jumlah nilai NaN per kolom:
Sumber    37204

Statistik deskriptif fitur numerik:
                          Tanggal      Harga     Tahun     Bulan      Hari  \
count                       76363   76363.00  76363.00  76363.00  76363.00   
mean   2024-08-30 05:05:21.375535   37891.96   2024.17      6.43     15.74   
min           2021-01-18 00:00:00    3000.00   2021.00      1.00      1.00   
25%           2023-08-19 00:00:00   15000.00   2023.00      4.00      8.00   
50%           2024-11-01 00:00:00   24000.00   2024.00      6.00     16.00   
75%           2025-12-09 00:00:00   40000.00   2025.00      9.00     23.00   
max           2026-11-10 00:00:00  440000.00   2026.00     12.00     31.00   
std                           NaN   46676.88      1.49      3.34      8.81   

       DayOfWeek   Quarter  WeekOfYear  Harga_Kemarin  Harga_Minggu_Lalu  \
count    76363.0  76363.00    76363.00       76363.00           76363.00   
mean         3.0   

## 💾 8. Export Dataset Final


In [7]:
# ── Export to CSV ─────────────────────────────────────────────────────────────
output_path: Path = FEATURES_DIR / "features_all_dataset.csv"
df.to_csv(output_path, index=False)
logger.info("✅ Dataset fitur final disimpan → %s", output_path.resolve())

print(f"\n{'═'*60}")
print(f"  📁 File    : features_all_dataset.csv")
print(f"  📊 Shape   : {df.shape}")
print(f"  📆 Rentang : {df['Tanggal'].min().date()} → {df['Tanggal'].max().date()}")
print(f"  🏷️  Komoditas: {df['Komoditas'].nunique()} unik")
print(f"  🔢 Fitur   : {df.shape[1]} kolom")
print(f"{'═'*60}")
df.tail(5)


10:44:33 [INFO] ✅ Dataset fitur final disimpan → C:\Users\Saputra Budiman\Documents\Tugas Kuliah Semester Genap 2025-2026\Sistem Informasi Manajemen\Skripsi - Pantau Pasar\SPK SIM\processed_data\features\features_all_dataset.csv



════════════════════════════════════════════════════════════
  📁 File    : features_all_dataset.csv
  📊 Shape   : (76363, 17)
  📆 Rentang : 2021-01-18 → 2026-11-10
  🏷️  Komoditas: 57 unik
  🔢 Fitur   : 17 kolom
════════════════════════════════════════════════════════════


,Tanggal,Komoditas,Harga,Sumber,Tahun,Bulan,Hari,DayOfWeek,Quarter,WeekOfYear,Harga_Kemarin,Harga_Minggu_Lalu,Rolling_Mean_7,Rolling_Mean_14,Rolling_Std_7,Rolling_Max_7,Rolling_Min_7
121975,2026-11-06,udang_basah,70000.0,NaN,2026,11,6,4,4,45,70000.0,70000.0,70000.0,70000.0,0.0,70000.0,70000.0
121976,2026-11-07,udang_basah,70000.0,NaN,2026,11,7,5,4,45,70000.0,70000.0,70000.0,70000.0,0.0,70000.0,70000.0
121977,2026-11-08,udang_basah,70000.0,NaN,2026,11,8,6,4,45,70000.0,70000.0,70000.0,70000.0,0.0,70000.0,70000.0
121978,2026-11-09,udang_basah,70000.0,NaN,2026,11,9,0,4,46,70000.0,70000.0,70000.0,70000.0,0.0,70000.0,70000.0
121979,2026-11-10,udang_basah,70000.0,NaN,2026,11,10,1,4,46,70000.0,70000.0,70000.0,70000.0,0.0,70000.0,70000.0
